# Workflows com LangGraph

Em um workflow a topologia do grafo está escrita no código, e o modelo decide o conteúdo de cada passo sem decidir o caminho. Este notebook monta os padrões dessa família: encadeamento, roteamento, paralelização, ciclo com verificação, subgrafos e orquestrador com trabalhadores.

In [ ]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai langgraph
# !pip install -q langchain-groq langchain-google-genai

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [ ]:
import operator
from typing import Annotated, Literal

from IPython.display import Image
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send

In [ ]:
model = init_chat_model("openai:gpt-4.1-mini", temperature=0.0)

TOPIC = "o elétron"

## Encadeamento

O encadeamento quebra a tarefa em passos, cada um com sua chamada, e passa a saída de um como entrada do seguinte. Entre dois passos cabe um gate: uma aresta condicional que confere o resultado do passo anterior e decide se o próximo precisa rodar.

In [ ]:
class ExplainState(TypedDict):
    topic: str
    explanation: str
    improved_explanation: str

In [ ]:
def generate_explanation(state: ExplainState) -> dict:
    """Escreve a primeira explicação do tópico."""
    msg = model.invoke(f"Explique {state['topic']} em duas frases, para quem não é da área.")
    return {"explanation": msg.content.strip()}


def verify_explanation(state: ExplainState) -> str:
    """Gate: verifica se a explicação traz algum número."""
    if any(char.isdigit() for char in state["explanation"]):
        return "Pass"
    return "Fail"


def add_data(state: ExplainState) -> dict:
    """Reescreve a explicação com um dado numérico concreto."""
    msg = model.invoke(f"Reescreva a explicação acrescentando um dado numérico concreto: {state['explanation']}")
    return {"improved_explanation": msg.content.strip()}

In [ ]:
workflow = StateGraph(ExplainState)

# nós
workflow.add_node("generate_explanation", generate_explanation)
workflow.add_node("add_data", add_data)

# arestas
workflow.add_edge(START, "generate_explanation")
workflow.add_conditional_edges(
    "generate_explanation", verify_explanation, {"Fail": "add_data", "Pass": END}
)
workflow.add_edge("add_data", END)

chain = workflow.compile()
Image(chain.get_graph().draw_mermaid_png())

In [ ]:
for topic in [TOPIC, "a velocidade da luz"]:
    state = chain.invoke({"topic": topic})
    print(topic, "|", verify_explanation(state), "|", sorted(state))
    print("  ", state.get("improved_explanation", state["explanation"])[:120])

O elétron reprovou no gate, porque a primeira explicação não trouxe número nenhum, e seguiu para o `add_data`. A velocidade da luz passou direto para o fim, e a chave `improved_explanation` nem existe no resultado: o gate serve para não pagar por um passo que a saída anterior já dispensa.

## Roteamento

O roteamento classifica a entrada e manda cada classe para um nó diferente. O nó classificador devolve um `Command`, que carrega no mesmo objeto a atualização do estado e o nome do destino, dispensando a aresta condicional.

In [ ]:
class AskState(TypedDict):
    question: str
    kind: str
    answer: str

In [ ]:
class Kind(BaseModel):
    kind: Literal["definicao", "numero", "opiniao"]


def classify(state: AskState) -> Command[Literal["explain", "estimate", "decline"]]:
    """Classifica a pergunta e segue direto para o nó correspondente."""
    kind = model.with_structured_output(Kind).invoke(state["question"]).kind
    goto = {"definicao": "explain", "numero": "estimate", "opiniao": "decline"}[kind]
    return Command(update={"kind": kind}, goto=goto)

In [ ]:
def explain(state: AskState) -> dict:
    """Responde uma definição em uma frase."""
    return {"answer": model.invoke(f"Em uma frase: {state['question']}").content}


def estimate(state: AskState) -> dict:
    """Responde com o número e a unidade."""
    return {"answer": model.invoke(f"Responda só com número e unidade: {state['question']}").content}


def decline(state: AskState) -> dict:
    """Recusa pedidos de opinião."""
    return {"answer": "Fora do escopo de uma nota técnica."}

In [ ]:
router_builder = StateGraph(AskState)

# nós
router_builder.add_node("classify", classify)
router_builder.add_node("explain", explain)
router_builder.add_node("estimate", estimate)
router_builder.add_node("decline", decline)

# arestas
router_builder.add_edge(START, "classify")

router = router_builder.compile()
Image(router.get_graph().draw_mermaid_png())

In [ ]:
questions = ["O que é um fóton?", "Qual a massa do elétron?", "Física é uma matéria difícil?"]

for question in questions:
    routed = router.invoke({"question": question})
    print(routed["kind"], "|", routed["answer"])

As três perguntas caíram em três nós diferentes, e nenhuma aresta partindo de `classify` foi declarada: as tracejadas do desenho vieram da anotação `Command[Literal[...]]`, que informa ao grafo quais nós podem ser alcançados a partir dali. O destino de cada execução veio do `goto`.

## Paralelização

Vários nós ligados ao mesmo nó de origem rodam em paralelo, e isso se chama distribuição e agregação. Cada ramo escreve em uma chave própria do estado, e um nó agregador junta os resultados depois que todos terminam.

In [ ]:
class StudyState(TypedDict):
    topic: str
    simple: str
    analogy: str
    question: str
    combined_output: str

In [ ]:
def explain_simply(state: StudyState) -> dict:
    """Explica o tópico para uma criança de dez anos."""
    msg = model.invoke(f"Explique {state['topic']} para uma criança de dez anos, em duas frases.")
    return {"simple": msg.content.strip()}


def make_analogy(state: StudyState) -> dict:
    """Dá uma analogia do dia a dia para o tópico."""
    msg = model.invoke(f"Dê uma analogia do dia a dia para {state['topic']}, em uma frase.")
    return {"analogy": msg.content.strip()}


def write_question(state: StudyState) -> dict:
    """Escreve uma pergunta de prova sobre o tópico."""
    msg = model.invoke(f"Escreva uma pergunta de prova sobre {state['topic']}, sem a resposta.")
    return {"question": msg.content.strip()}

In [ ]:
def aggregator(state: StudyState) -> dict:
    """Reúne as três saídas em um texto só, sem chamar o modelo."""
    combined = f"Material de estudo sobre {state['topic']}\n\n"
    combined += f"EM TERMOS SIMPLES:\n{state['simple']}\n\n"
    combined += f"ANALOGIA:\n{state['analogy']}\n\n"
    combined += f"PERGUNTA:\n{state['question']}"
    return {"combined_output": combined}

In [ ]:
parallel_builder = StateGraph(StudyState)

# nós
parallel_builder.add_node("explain_simply", explain_simply)
parallel_builder.add_node("make_analogy", make_analogy)
parallel_builder.add_node("write_question", write_question)
parallel_builder.add_node("aggregator", aggregator)

# arestas
parallel_builder.add_edge(START, "explain_simply")
parallel_builder.add_edge(START, "make_analogy")
parallel_builder.add_edge(START, "write_question")
parallel_builder.add_edge("explain_simply", "aggregator")
parallel_builder.add_edge("make_analogy", "aggregator")
parallel_builder.add_edge("write_question", "aggregator")
parallel_builder.add_edge("aggregator", END)

parallel_workflow = parallel_builder.compile()

In [ ]:
Image(parallel_workflow.get_graph().draw_mermaid_png())

In [ ]:
state = parallel_workflow.invoke({"topic": TOPIC})

print(state["combined_output"])

Cada ramo escreveu em uma chave própria do estado, então nenhuma escrita concorreu com outra e o estado dispensa regra de combinação. O `aggregator` é o único nó que não chama o modelo: ele espera os três ramos terminarem e monta o texto com o que encontrou no estado.

### Exercício 1

O estado e os nós de um grafo que monta a ficha de uma notícia estão prontos. Monte o grafo com `write_headline`, `summarize` e `extract_tags` em paralelo, todos seguidos de `build_card`. Compile como `card_graph` e rode com `ARTICLE`.

In [ ]:
ARTICLE = """O Instituto Nacional de Meteorologia emitiu alerta de chuvas intensas para o litoral do Rio Grande do Norte
entre quinta e sábado. A previsão é de até 80 milímetros por dia em Natal, com risco de alagamentos em áreas baixas.
A Defesa Civil recomenda evitar deslocamentos durante as pancadas mais fortes e abriu um telefone para ocorrências."""


class CardState(TypedDict):
    article: str
    headline: str
    summary: str
    tags: str
    card: str


def write_headline(state: CardState) -> dict:
    """Escreve uma manchete para a notícia."""
    msg = model.invoke(f"Escreva uma manchete de até 60 caracteres. Responda só com a manchete.\n{state['article']}")
    return {"headline": msg.content.strip()}


def summarize(state: CardState) -> dict:
    """Resume a notícia em uma frase."""
    msg = model.invoke(f"Resuma a notícia em uma frase.\n{state['article']}")
    return {"summary": msg.content.strip()}


def extract_tags(state: CardState) -> dict:
    """Lista três palavras-chave da notícia."""
    msg = model.invoke(f"Liste três palavras-chave, separadas por vírgula. Responda só com elas.\n{state['article']}")
    return {"tags": msg.content.strip()}


def build_card(state: CardState) -> dict:
    """Reúne as três partes na ficha, sem chamar o modelo."""
    return {"card": f"{state['headline']}\n{state['summary']}\nTags: {state['tags']}"}

In [ ]:
# Seu código aqui


Image(card_graph.get_graph().draw_mermaid_png())

## Ciclo com verificação

Uma aresta pode voltar para um nó anterior, e aí o grafo tem um laço. Um nó escreve, outro verifica as regras e aponta a primeira que falhou, e o texto volta para a escrita enquanto houver problema. As correções se acumulam no estado, então cada volta atende a uma regra nova sem esquecer as anteriores. O verificador é escrito em código, o que o deixa conferir as regras sem errar e sem gastar token.

In [ ]:
class DraftState(TypedDict):
    topic: str
    note: str
    feedback: list[str]
    approved: bool
    attempts: int

In [ ]:
def rewrite(state: DraftState) -> dict:
    """Escreve o parágrafo, incorporando as correções acumuladas."""
    request = f"Escreva um parágrafo curto sobre {state['topic']}. Responda só com o parágrafo."
    if state.get("feedback"):
        request += f"\n\nVersão anterior:\n{state['note']}\n\nCorreções: {'; '.join(state['feedback'])}"
    return {"note": model.invoke(request).content.strip(), "attempts": state.get("attempts", 0) + 1}

In [ ]:
def review(state: DraftState) -> dict:
    """Confere as regras em ordem e acrescenta a primeira que falhou às correções."""
    note = state["note"]
    if not any(char.isdigit() for char in note):
        problem = "inclua um número"
    elif "partícula" in note.lower():
        problem = "não use a palavra partícula"
    elif not note.endswith("?"):
        problem = "termine com uma pergunta"
    else:
        return {"approved": True}
    return {"approved": False, "feedback": state.get("feedback", []) + [problem]}


def again(state: DraftState) -> Literal["rewrite", END]:
    """Reescreve enquanto houver problema e restar tentativa."""
    if state["approved"] or state["attempts"] >= 5:
        return END
    return "rewrite"

In [ ]:
writer_builder = StateGraph(DraftState)

# nós
writer_builder.add_node("rewrite", rewrite)
writer_builder.add_node("review", review)

# arestas
writer_builder.add_edge(START, "rewrite")
writer_builder.add_edge("rewrite", "review")
writer_builder.add_conditional_edges("review", again, ["rewrite", END])

writer = writer_builder.compile()
Image(writer.get_graph().draw_mermaid_png())

In [ ]:
for step in writer.stream({"topic": TOPIC}, stream_mode="updates"):
    node, update = next(iter(step.items()))
    print(node, "|", update)

Cada volta atendeu à correção que o revisor apontou, e a lista `feedback` cresceu uma regra por volta até o `approved` liberar a saída. O `attempts` é o limite do laço: sem ele, uma regra que o modelo nunca satisfaz roda para sempre, e com ele o grafo entrega o último rascunho mesmo reprovado.

### Exercício 2

O estado e as funções de um grafo que escreve um tweet sobre `TOPIC` estão prontos. A função `check` aponta a primeira regra que o tweet quebra, e o `write_tweet` acumula essas correções. Monte o grafo com o nó `write_tweet` e uma aresta condicional com `retry`, que volta para o próprio `write_tweet` enquanto houver problema. Compile como `tweet_graph` e rode.

In [ ]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    feedback: list[str]
    attempts: int


def check(tweet: str) -> str:
    """Devolve a primeira regra que o tweet quebra, ou vazio se ele passar."""
    if "#" not in tweet:
        return "inclua uma hashtag"
    if not any(char.isdigit() for char in tweet):
        return "inclua um número"
    if len(tweet) > 120:
        return "use no máximo 120 caracteres"
    return ""


def write_tweet(state: TweetState) -> dict:
    """Escreve o tweet, somando a correção do anterior às que já havia."""
    feedback = state.get("feedback", [])
    request = f"Escreva um tweet sobre {state['topic']}. Responda só com o tweet."
    if state.get("tweet"):
        feedback = feedback + [check(state["tweet"])]
        request += f"\n\nVersão anterior:\n{state['tweet']}\n\nCorreções: {'; '.join(feedback)}"
    return {"tweet": model.invoke(request).content.strip(), "feedback": feedback, "attempts": state.get("attempts", 0) + 1}


def retry(state: TweetState) -> Literal["write_tweet", END]:
    """Reescreve enquanto o tweet quebrar alguma regra e restar tentativa."""
    if not check(state["tweet"]) or state["attempts"] >= 5:
        return END
    return "write_tweet"

In [ ]:
# Seu código aqui


Image(tweet_graph.get_graph().draw_mermaid_png())

## Subgrafos

Um grafo compilado é um runnable, e por isso pode entrar como nó de outro grafo. As chaves em comum entre os dois estados são o que atravessa a fronteira, e as chaves que só existem no subgrafo ficam contidas nele.

In [ ]:
class PageState(TypedDict):
    topic: str
    note: str
    headline: str

In [ ]:
def headline(state: PageState) -> dict:
    """Escreve a manchete a partir do parágrafo aprovado."""
    request = f"Escreva uma manchete de até 50 caracteres para:\n{state['note']}"
    return {"headline": model.invoke(request).content.strip()}

In [ ]:
page_builder = StateGraph(PageState)

# nós, e o grafo compilado entra como um deles
page_builder.add_node("improve", writer)
page_builder.add_node("headline", headline)

# arestas
page_builder.add_edge(START, "improve")
page_builder.add_edge("improve", "headline")

page = page_builder.compile()
Image(page.get_graph(xray=1).draw_mermaid_png())

In [ ]:
published = page.invoke({"topic": TOPIC})

print(published["headline"])
print(sorted(published))

O `xray=1` desenha o subgrafo aberto dentro do nó `improve`. As chaves `feedback`, `approved` e `attempts` não aparecem no resultado porque não fazem parte do `PageState`: o ciclo de revisão as usou e as deixou para trás.

## Orquestrador com trabalhadores

Quando a quantidade de ramos só se conhece em tempo de execução, as arestas não cabem no código. O `Send` resolve isso: um nó devolve uma lista de `Send`, cada um com o nome do nó de destino e o estado daquele ramo.

In [ ]:
class ReportState(TypedDict):
    topic: str
    sections: list[str]
    parts: Annotated[list, operator.add]


class SectionState(TypedDict):
    topic: str
    section: str

In [ ]:
class Plan(BaseModel):
    sections: list[str] = Field(description="Títulos das seções, de três a quatro")


def plan(state: ReportState) -> dict:
    """Decide as seções do relatório."""
    request = f"Liste as seções de um relatório curto sobre {state['topic']}."
    return {"sections": model.with_structured_output(Plan).invoke(request).sections}

In [ ]:
def dispatch(state: ReportState) -> list[Send]:
    """Abre um trabalhador por seção planejada."""
    return [Send("write_section", {"topic": state["topic"], "section": title}) for title in state["sections"]]


def write_section(state: SectionState) -> dict:
    """Escreve uma seção em duas frases."""
    request = f"Escreva duas frases sobre {state['section']}, em um relatório sobre {state['topic']}."
    return {"parts": [f"{state['section']}: {model.invoke(request).content}"]}

In [ ]:
report_builder = StateGraph(ReportState)

# nós
report_builder.add_node("plan", plan)
report_builder.add_node("write_section", write_section)

# arestas
report_builder.add_edge(START, "plan")
report_builder.add_conditional_edges("plan", dispatch, ["write_section"])
report_builder.add_edge("write_section", END)

orchestrator = report_builder.compile()
Image(orchestrator.get_graph().draw_mermaid_png())

In [ ]:
report = orchestrator.invoke({"topic": TOPIC})

print(len(report["parts"]))
for part in report["parts"]:
    print(part)

A quantidade de trabalhadores veio do `plan`, e o `SectionState` mostra que cada ramo recebeu um estado próprio em vez do estado do grafo. O desenho traz uma caixa só para `write_section`, porque a quantidade de cópias não existe no momento da compilação.

### Exercício 3

Monte um grafo que abre um trabalhador por documento de `DOCUMENTS` com `Send`. Cada trabalhador classifica o documento como nota, ata ou reclamação e devolve o tipo por um reducer. Desenhe o grafo e rode.

In [ ]:
DOCUMENTS = [
    "Nota fiscal 4471. Fornecedor Papelaria Central. Total de R$ 1.280,00 em 12/03/2026.",
    "Ata da reunião de 05/03/2026. O colegiado aprovou a compra de dois projetores e adiou a reforma do laboratório.",
    "Comprei a cafeteira no dia 3 e ela parou de esquentar na primeira semana. Quero uma solução.",
    "Nota fiscal 4472. Fornecedor Gráfica Norte. Total de R$ 345,50 em 18/03/2026.",
    "Ata da reunião de 19/03/2026. O colegiado aprovou o calendário de defesas e criou a comissão de estágio.",
    "A impressora chegou com o painel trincado e a assistência não responde há duas semanas.",
]

In [ ]:
# Seu código aqui